# 🧠 Reconocimiento de dígitos y evaluación del modelo

Este notebook desarrolla de principio a fin un ejemplo introductorio de Machine Learning: enseñar a una máquina a reconocer dígitos escritos a mano.

Se utilizará el dataset `digits` de scikit-learn, una división 80/20 y una regresión logística multiclase. La evaluación incluirá Accuracy, Precision, Recall, F1 Score, matriz de confusión, curva ROC y AUC.

## 1. Introducción

Machine Learning permite construir modelos que aprenden patrones a partir de datos. En este ejercicio, cada imagen de un dígito se convierte en valores numéricos y el modelo aprende a relacionarlos con una etiqueta entre 0 y 9.

Además de comprobar si el modelo acierta, evaluaremos su calidad con varias métricas. Para ROC/AUC utilizaremos la estrategia **One-vs-Rest (OvR)**, adecuada para transformar el problema multiclase en varios problemas binarios.

In [ ]:
# Importamos NumPy para trabajar con arreglos numéricos y crear el listado de clases.
import numpy as np  # NumPy permite realizar operaciones numéricas de forma eficiente.
# Importamos Pandas para presentar resultados en tablas.
import pandas as pd  # Pandas facilita trabajar con datos tabulares y exportarlos a CSV.
# Importamos Matplotlib para visualizar imágenes y gráficas.
import matplotlib.pyplot as plt  # Matplotlib permite crear las visualizaciones del ejercicio.
# Importamos el dataset de dígitos incluido en scikit-learn.
from sklearn.datasets import load_digits  # load_digits proporciona imágenes pequeñas de dígitos escritos a mano.
# Importamos la función para dividir los datos en entrenamiento y prueba.
from sklearn.model_selection import train_test_split  # Permite separar datos para entrenar y evaluar.
# Importamos el algoritmo de clasificación elegido.
from sklearn.linear_model import LogisticRegression  # La regresión logística es sencilla y permite obtener probabilidades.
# Importamos las métricas necesarias para evaluar el modelo.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score  # Calculamos métricas globales.
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score  # Calculamos matriz, reporte, ROC y AUC.
# Importamos la función para transformar las etiquetas multiclase en formato binario.
from sklearn.preprocessing import label_binarize  # Es necesaria para calcular ROC One-vs-Rest.

## 3. Carga del dataset

El dataset `digits` contiene 1797 imágenes. Cada imagen tiene una resolución de 8 × 8 píxeles, por lo que el modelo recibe 64 características por ejemplo. Las etiquetas son los dígitos del 0 al 9.

In [ ]:
# Cargamos el dataset real de dígitos.
digits = load_digits()  # Obtenemos las imágenes, características y etiquetas del conjunto educativo.
# Extraemos las características numéricas y las etiquetas.
X, y = digits.data, digits.target  # X contiene los píxeles y y contiene el dígito correcto.
# Mostramos la cantidad total de ejemplos.
print("Cantidad de muestras:", X.shape[0])  # El primer valor de shape indica cuántas imágenes existen.
# Mostramos la cantidad de características.
print("Cantidad de características:", X.shape[1])  # Cada imagen está representada por 64 valores de píxel.
# Mostramos las clases disponibles.
print("Clases:", np.unique(y))  # Las clases son los dígitos del 0 al 9.
# Mostramos las dimensiones originales de las imágenes.
print("Dimensión de cada imagen:", digits.images.shape[1:])  # Cada imagen tiene 8 filas por 8 columnas.

## 4. Exploración de los datos

Las características son intensidades de píxel. En este dataset no es obligatorio realizar un escalado para poder entrenar el modelo; por ello se mantiene el flujo simple y se evita añadir una transformación que no sea imprescindible. La división de entrenamiento y prueba sí es necesaria para medir generalización.

In [ ]:
# Creamos una tabla con las primeras muestras para inspeccionar los valores de píxel.
df = pd.DataFrame(X)  # Convertimos las 64 características en columnas de una tabla.
# Añadimos la etiqueta real de cada imagen.
df["Digito"] = y  # La nueva columna indica qué dígito representa cada muestra.
# Mostramos las primeras cinco filas.
display(df.head())  # Permite comprobar cómo se representan numéricamente las imágenes.
# Mostramos la distribución de las clases.
print(y.size, "etiquetas en total")  # Confirmamos la cantidad total de etiquetas.
print(pd.Series(y).value_counts().sort_index())  # Contamos cuántas muestras existen de cada dígito.

## 5. Visualización de los dígitos

Cada imagen es una matriz de 8 × 8 valores. Cada valor representa la intensidad de un píxel. Visualizar las matrices como imágenes permite relacionar directamente los datos numéricos con el dígito real.

In [ ]:
# Creamos una figura con varios ejemplos del dataset.
fig, axes = plt.subplots(2, 5, figsize=(10, 5))  # Organizamos diez imágenes en dos filas de cinco.
# Recorremos las posiciones de la figura para mostrar diez ejemplos.
for ax, image, label in zip(axes.ravel(), digits.images[:10], y[:10]):  # Asociamos cada eje con una imagen y su etiqueta.
    ax.imshow(image, cmap="gray")  # Mostramos la matriz de píxeles como una imagen en escala de grises.
    ax.set_title(f"Dígito real: {label}")  # Indicamos la etiqueta correcta sobre cada imagen.
    ax.axis("off")  # Ocultamos los ejes porque aquí interesa observar la forma del dígito.
plt.tight_layout()  # Ajustamos automáticamente los espacios entre imágenes.
plt.show()  # Presentamos la figura en el notebook.

## 6. Preparación y división de datos

Utilizamos 80 % de los ejemplos para entrenamiento y 20 % para prueba. `stratify=y` conserva aproximadamente la misma proporción de cada dígito en ambos grupos y `random_state=42` hace reproducible la división.

In [ ]:
# Dividimos las características y etiquetas en entrenamiento y prueba.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)  # Reservamos 20 % para evaluar.
# Mostramos el tamaño del conjunto de entrenamiento.
print("Entrenamiento:", X_train.shape[0], "muestras")  # Estas muestras se utilizan para aprender.
# Mostramos el tamaño del conjunto de prueba.
print("Prueba:", X_test.shape[0], "muestras")  # Estas muestras se mantienen separadas hasta la evaluación.

## 7. Entrenamiento

Se utiliza `LogisticRegression`. Es un modelo sencillo de explicar y apropiado para clasificación multiclase. Además, `predict_proba` proporciona probabilidades por clase, que necesitamos para construir ROC y calcular AUC.

In [ ]:
# Creamos el clasificador de regresión logística.
model = LogisticRegression(max_iter=2000, solver="lbfgs")  # Aumentamos las iteraciones para favorecer la convergencia.
# Entrenamos el modelo exclusivamente con el conjunto de entrenamiento.
model.fit(X_train, y_train)  # El modelo aprende patrones que relacionan píxeles con dígitos.

## 8. Predicciones

Las predicciones se realizan sobre el conjunto de prueba. También guardamos las probabilidades para posteriormente evaluar ROC/AUC.

In [ ]:
# Generamos la predicción de clase para cada ejemplo de prueba.
y_pred = model.predict(X_test)  # El modelo selecciona el dígito con mayor probabilidad estimada.
# Obtenemos las probabilidades de las diez clases.
y_score = model.predict_proba(X_test)  # Estas probabilidades permiten variar el threshold para ROC.
# Mostramos cinco predicciones reales del conjunto de prueba.
for index in range(5):  # Recorremos los primeros cinco ejemplos de prueba.
    result = "Correcto" if y_test[index] == y_pred[index] else "Incorrecto"  # Comparamos la etiqueta real con la predicción.
    print(f"Dígito real: {y_test[index]} | Predicción: {y_pred[index]} | Resultado: {result}")  # Presentamos el resultado.

## 9. Matriz de confusión

La matriz de confusión cruza el dígito real con el dígito predicho. La diagonal representa aciertos; los valores fuera de la diagonal representan errores.

In [ ]:
# Calculamos la matriz de confusión.
cm = confusion_matrix(y_test, y_pred)  # Cada fila representa la clase real y cada columna la clase predicha.
# Creamos una figura para visualizar la matriz.
plt.figure(figsize=(8, 6))  # Definimos un tamaño adecuado para diez clases.
# Mostramos la matriz como una imagen.
plt.imshow(cm, interpolation="nearest")  # Cada celda representa la cantidad de ejemplos.
# Añadimos el título.
plt.title("Matriz de confusión - Reconocimiento de dígitos")  # Identificamos claramente la gráfica.
# Añadimos una barra de referencia de valores.
plt.colorbar()  # Facilita interpretar la intensidad de cada celda.
# Etiquetamos las clases del eje horizontal.
plt.xticks(np.arange(10), np.arange(10))  # Mostramos los dígitos predichos.
# Etiquetamos las clases del eje vertical.
plt.yticks(np.arange(10), np.arange(10))  # Mostramos los dígitos reales.
# Identificamos el eje horizontal.
plt.xlabel("Dígito predicho")  # Indica qué representan las columnas.
# Identificamos el eje vertical.
plt.ylabel("Dígito real")  # Indica qué representan las filas.
# Calculamos un umbral para decidir el color del texto.
threshold = cm.max() / 2  # Los valores altos necesitan texto claro para conservar legibilidad.
# Recorremos todas las celdas de la matriz.
for i in range(cm.shape[0]):  # Recorremos las filas de las diez clases.
    for j in range(cm.shape[1]):  # Recorremos las columnas de las diez clases.
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > threshold else "black")  # Escribimos cada valor.
# Ajustamos los espacios de la figura.
plt.tight_layout()  # Evitamos que las etiquetas queden cortadas.
# Mostramos la gráfica.
plt.show()  # Presentamos la matriz al usuario.

## 10. Métricas

- **Accuracy:** proporción total de predicciones correctas.
- **Precision:** qué tan fiables son las predicciones positivas.
- **Recall:** qué proporción de los casos reales se recupera.
- **F1 Score:** equilibrio entre Precision y Recall.

Se utiliza promedio ponderado para las métricas globales, teniendo en cuenta el número de ejemplos de cada clase.

In [ ]:
# Calculamos accuracy.
accuracy = accuracy_score(y_test, y_pred)  # Mide la proporción total de aciertos.
# Calculamos precision ponderada.
precision = precision_score(y_test, y_pred, average="weighted")  # Ponderamos cada clase según su cantidad de ejemplos.
# Calculamos recall ponderado.
recall = recall_score(y_test, y_pred, average="weighted")  # Ponderamos cada clase según su soporte.
# Calculamos F1 ponderado.
f1 = f1_score(y_test, y_pred, average="weighted")  # Combina precision y recall.
# Mostramos las métricas.
print(f"Accuracy: {accuracy:.4f}")  # Mostramos accuracy con cuatro decimales.
print(f"Precision ponderada: {precision:.4f}")  # Mostramos precision.
print(f"Recall ponderado: {recall:.4f}")  # Mostramos recall.
print(f"F1 Score ponderado: {f1:.4f}")  # Mostramos F1.

## 11. F1 Score por clase

La tabla siguiente se calcula directamente a partir de las predicciones reales. `support` indica cuántos ejemplos de cada dígito existen en el conjunto de prueba.

In [ ]:
# Generamos un informe completo de clasificación.
report = classification_report(y_test, y_pred, output_dict=True)  # Calculamos precision, recall, F1 y support por clase.
# Convertimos el informe en DataFrame y seleccionamos únicamente las diez clases.
f1_table = pd.DataFrame(report).T.loc[[str(i) for i in range(10)], ["precision", "recall", "f1-score", "support"]].reset_index()  # Organizamos la tabla.
# Cambiamos el nombre de la primera columna.
f1_table = f1_table.rename(columns={"index": "Digito"})  # Hacemos más clara la tabla para la presentación.
# Mostramos la tabla por clase.
display(f1_table)  # Presentamos los valores calculados automáticamente.

## 12. Curva ROC y AUC

ROC representa **TPR (True Positive Rate)** frente a **FPR (False Positive Rate)** para distintos thresholds.

- Un **threshold** es el umbral utilizado para decidir si una probabilidad pertenece a una clase.
- La diagonal representa el comportamiento aproximado de un clasificador aleatorio.
- Una curva cercana a la esquina superior izquierda indica buena capacidad de discriminación.

Como hay diez clases, usamos **One-vs-Rest (OvR)**: cada dígito se compara contra los otros nueve.

In [ ]:
# Convertimos las etiquetas reales en diez columnas binarias.
y_test_bin = label_binarize(y_test, classes=np.arange(10))  # Cada columna representa una clase frente a todas las demás.
# Calculamos el AUC macro One-vs-Rest.
auc_macro = roc_auc_score(y_test_bin, y_score, average="macro", multi_class="ovr")  # Damos el mismo peso a cada dígito.
# Creamos la figura ROC.
plt.figure(figsize=(10, 8))  # Definimos espacio suficiente para diez curvas y su leyenda.
# Recorremos las diez clases.
for class_id in range(10):  # Calculamos una curva ROC independiente para cada dígito.
    fpr, tpr, _ = roc_curve(y_test_bin[:, class_id], y_score[:, class_id])  # Obtenemos FPR y TPR para distintos thresholds.
    class_auc = auc(fpr, tpr)  # Calculamos el área bajo la curva de esa clase.
    plt.plot(fpr, tpr, label=f"Clase {class_id} (AUC = {class_auc:.4f})")  # Dibujamos y etiquetamos la curva.
# Dibujamos la diagonal de referencia.
plt.plot([0, 1], [0, 1], linestyle="--", label="Clasificador aleatorio")  # Esta línea representa una referencia sin capacidad discriminativa.
# Añadimos el título incluyendo el AUC macro.
plt.title(f"Curva ROC multiclase - AUC macro OvR = {auc_macro:.4f}")  # Mostramos el resultado global.
# Etiquetamos el eje X.
plt.xlabel("False Positive Rate")  # FPR representa la tasa de falsos positivos.
# Etiquetamos el eje Y.
plt.ylabel("True Positive Rate")  # TPR representa la tasa de verdaderos positivos.
# Añadimos la leyenda.
plt.legend(loc="lower right", fontsize=8)  # Identificamos cada clase y su AUC.
# Añadimos una cuadrícula ligera.
plt.grid(True, alpha=0.25)  # Facilita leer coordenadas de la gráfica.
# Ajustamos los espacios.
plt.tight_layout()  # Evitamos que la leyenda o etiquetas se corten.
# Mostramos la curva.
plt.show()  # Presentamos la gráfica ROC.

## 13. Tabla ROC

La siguiente tabla contiene los valores que salen directamente del cálculo de `roc_curve`: clase, FPR, TPR y threshold. Estos son los datos utilizados para construir las curvas.

In [ ]:
# Creamos una lista donde almacenaremos todos los puntos ROC.
roc_rows = []  # Cada elemento representará un punto de una clase.
# Recorremos las diez clases.
for class_id in range(10):  # Calculamos ROC de cada dígito contra el resto.
    fpr, tpr, thresholds = roc_curve(y_test_bin[:, class_id], y_score[:, class_id])  # Obtenemos los puntos reales del modelo.
    # Recorremos simultáneamente FPR, TPR y threshold.
    for fpr_value, tpr_value, threshold in zip(fpr, tpr, thresholds):  # Asociamos cada threshold con su punto ROC.
        roc_rows.append({"Clase": class_id, "FPR": fpr_value, "TPR": tpr_value, "Threshold": threshold})  # Guardamos el resultado.
# Convertimos la lista en una tabla.
roc_table = pd.DataFrame(roc_rows)  # Pandas facilita visualizar y exportar los datos.
# Mostramos una parte de la tabla porque puede contener muchos puntos.
display(roc_table.head(20))  # Los datos completos se conservan en el CSV.

## 14. Tabla final de rendimiento y conclusión

Los siguientes valores son los resultados reales obtenidos con la división reproducible 80/20 y la regresión logística.

In [ ]:
# Creamos una tabla resumen con las métricas principales.
metrics_table = pd.DataFrame({"Metrica": ["Accuracy", "Precision", "Recall", "F1 Score", "AUC Macro OvR"], "Resultado": [accuracy, precision, recall, f1, auc_macro]})  # Organizamos los resultados finales.
# Mostramos la tabla.
display(metrics_table)  # Presentamos las métricas de forma clara.
# Indicamos el resultado global del experimento.
print(f"El modelo obtuvo Accuracy={accuracy:.4f}, F1 ponderado={f1:.4f} y AUC macro OvR={auc_macro:.4f}.")  # Resumimos el rendimiento real.

## 15. Guardar resultados

En Google Colab se pueden descargar posteriormente los archivos generados o subirlos a GitHub. Los CSV contienen las métricas y los datos ROC, mientras que las imágenes contienen la matriz de confusión y la curva ROC.

In [ ]:
# Creamos la carpeta de resultados.
import os  # Importamos os para crear carpetas y construir rutas de archivos.
os.makedirs("resultados", exist_ok=True)  # Creamos la carpeta si todavía no existe.
# Guardamos la tabla F1.
f1_table.to_csv("resultados/f1_score.csv", index=False)  # Exportamos las métricas por clase sin guardar el índice.
# Guardamos la tabla ROC.
roc_table.to_csv("resultados/roc.csv", index=False)  # Exportamos todos los puntos calculados.
# Guardamos la tabla de métricas globales.
metrics_table.to_csv("resultados/metricas_modelo.csv", index=False)  # Exportamos el resumen final.
# Creamos la carpeta de gráficas.
os.makedirs("graficas", exist_ok=True)  # Creamos la carpeta destinada a imágenes.
# Guardamos la curva ROC.
plt.figure(figsize=(10, 8))  # Creamos una nueva figura para guardar la curva.
for class_id in range(10):  # Recorremos cada clase.
    fpr, tpr, _ = roc_curve(y_test_bin[:, class_id], y_score[:, class_id])  # Recuperamos los puntos ROC.
    class_auc = auc(fpr, tpr)  # Calculamos AUC de la clase.
    plt.plot(fpr, tpr, label=f"Clase {class_id} (AUC = {class_auc:.4f})")  # Dibujamos la curva.
plt.plot([0, 1], [0, 1], linestyle="--", label="Clasificador aleatorio")  # Añadimos la diagonal de referencia.
plt.title(f"Curva ROC multiclase - AUC macro OvR = {auc_macro:.4f}")  # Añadimos el título.
plt.xlabel("False Positive Rate")  # Nombramos el eje X.
plt.ylabel("True Positive Rate")  # Nombramos el eje Y.
plt.legend(loc="lower right", fontsize=8)  # Añadimos la leyenda.
plt.grid(True, alpha=0.25)  # Añadimos una cuadrícula ligera.
plt.tight_layout()  # Ajustamos la figura.
plt.savefig("graficas/curva_roc.png", dpi=200)  # Guardamos la curva como imagen PNG.
plt.show()  # Mostramos la imagen guardada en el notebook.

## 16. ¿Qué aprendí?

- Una imagen puede convertirse en características numéricas.
- Una característica puede representar la intensidad de un píxel.
- Una etiqueta indica la clase correcta.
- Entrenar significa aprender patrones a partir de ejemplos.
- Predecir significa utilizar esos patrones con datos nuevos.
- F1 combina Precision y Recall.
- ROC permite estudiar el comportamiento del clasificador al cambiar el threshold.
- AUC resume la capacidad de discriminación.
- La matriz de confusión permite observar qué clases se confunden.


## 17. Preguntas que podría hacer el profesor

1. ¿Qué dataset utilizaste? — `digits` de scikit-learn.
2. ¿Qué representa cada píxel? — La intensidad de uno de los 64 píxeles.
3. ¿Qué es Machine Learning? — Aprendizaje de patrones a partir de datos para realizar predicciones.
4. ¿Qué significa entrenar? — Ajustar el modelo usando datos conocidos.
5. ¿Por qué separar entrenamiento y prueba? — Para evaluar generalización con datos no usados durante el aprendizaje.
6. ¿Qué es Precision? — Qué proporción de predicciones positivas son correctas.
7. ¿Qué es Recall? — Qué proporción de casos reales se identifica correctamente.
8. ¿Qué es F1? — Una combinación equilibrada de Precision y Recall.
9. ¿Qué es ROC? — Una curva de TPR frente a FPR para distintos thresholds.
10. ¿Qué es AUC? — Área bajo la curva ROC.
11. ¿Qué es FPR? — Tasa de falsos positivos.
12. ¿Qué es TPR? — Tasa de verdaderos positivos.
13. ¿Por qué es multiclase? — Porque existen diez clases, del 0 al 9.
14. ¿Cómo sabes que funciona bien? — Por las métricas calculadas en el conjunto de prueba y la matriz de confusión.
15. ¿Qué mejorarías? — Comparar modelos, ajustar hiperparámetros y evaluar datos externos.
